In [1]:
import pandas as pd
import seaborn as sns
import plotly.express as px
import plotly.io as pio

# ⭐ 무조건 브라우저로 출력 (환경 문제 해결)
pio.renderers.default = "browser"


# -----------------------------
# 1. 데이터 타입 자동 분류 (개선)
# -----------------------------
def classify_column(df, col):
    if pd.api.types.is_numeric_dtype(df[col]):
        unique_ratio = df[col].nunique() / len(df)

        if unique_ratio < 0.05:
            return "이산형"
        else:
            return "연속형"

    else:
        ordered_candidates = ["low", "medium", "high", "1st", "2nd", "3rd"]

        values = df[col].dropna().astype(str).str.lower().unique()

        if any(v in ordered_candidates for v in values):
            return "순서형"
        else:
            return "명목형"


# -----------------------------
# 2. 단일 변수 그래프 추천 (안전)
# -----------------------------
def recommend_single_plot(col_type):
    mapping = {
        "연속형": "histogram",
        "이산형": "bar",
        "명목형": "count",
        "순서형": "bar"
    }
    return mapping.get(col_type, "histogram")


# -----------------------------
# 3. 두 변수 그래프 추천 (개선)
# -----------------------------
def recommend_pair_plot(type1, type2):
    if type1 == "연속형" and type2 == "연속형":
        return "scatter"
    elif type1 in ["명목형", "순서형"] and type2 == "연속형":
        return "box"
    elif type2 in ["명목형", "순서형"] and type1 == "연속형":
        return "box"
    else:
        return "bar"


# -----------------------------
# 4. 자동 시각화 (완성)
# -----------------------------
def auto_plot(df, col1, col2=None):
    type1 = classify_column(df, col1)

    # -------------------
    # 단일 변수
    # -------------------
    if col2 is None:
        plot_type = recommend_single_plot(type1)

        if plot_type == "histogram":
            fig = px.histogram(df, x=col1)

        elif plot_type == "count":
            fig = px.histogram(df, x=col1)

        elif plot_type == "bar":
            counts = df[col1].value_counts().reset_index()
            counts.columns = [col1, "count"]
            fig = px.bar(counts, x=col1, y="count")

        else:
            fig = px.histogram(df, x=col1)

        fig.show()
        print(f"{col1} ({type1}) → {plot_type}")

    # -------------------
    # 두 변수
    # -------------------
    else:
        type2 = classify_column(df, col2)
        plot_type = recommend_pair_plot(type1, type2)

        if plot_type == "scatter":
            fig = px.scatter(df, x=col1, y=col2)

        elif plot_type == "box":
            # 범주형 vs 수치형 자동 정렬
            if type1 in ["명목형", "순서형"]:
                fig = px.box(df, x=col1, y=col2)
            else:
                fig = px.box(df, x=col2, y=col1)

        else:
            grouped = df.groupby(col1)[col2].mean().reset_index()
            fig = px.bar(grouped, x=col1, y=col2)

        fig.show()
        print(f"{col1} ({type1}) vs {col2} ({type2}) → {plot_type}")

In [3]:
df = sns.load_dataset("iris")

auto_plot(df, "sepal_length")
auto_plot(df, "petal_length", "petal_width")
auto_plot(df, "species", "petal_length")


sepal_length (연속형) → histogram
petal_length (연속형) vs petal_width (연속형) → scatter
species (명목형) vs petal_length (연속형) → box


In [5]:
df = sns.load_dataset("titanic")

auto_plot(df, "age")
auto_plot(df, "sex")
auto_plot(df, "sex", "fare")

age (연속형) → histogram
sex (명목형) → count
sex (명목형) vs fare (연속형) → box


In [6]:
df = sns.load_dataset("tips")

auto_plot(df, "total_bill")
auto_plot(df, "tip", "total_bill")
auto_plot(df, "day", "tip")

total_bill (연속형) → histogram
tip (연속형) vs total_bill (연속형) → scatter
day (명목형) vs tip (연속형) → box
